# Train and Validate PD Models from Kaggle

This notebook demonstrates the main CRVK workflow after training a binary PD model:

1. Download a public Kaggle credit-risk dataset.
2. Split the data into train and test.
3. Train two simple models: Logistic Regression and XGBoost.
4. Generate scored test samples with `target`, `pd`, `score`, and a segment column.
5. Create one HTML training-validation report per model using `PDValidationSuite.run(validation_data=...)`.

The reports are written to `reports/notebooks/kaggle_training_validation/`.


## 1. Imports and paths

The notebook is designed to run from either the repository root or the `examples/` directory. It stores raw Kaggle files under `data/raw/`, which is ignored by git.


In [ ]:
from __future__ import annotations

import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

from credit_risk_validation import PDValidationSuite

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "examples":
    REPO_ROOT = REPO_ROOT.parent

DATASET_SLUG = "uciml/default-of-credit-card-clients-dataset"
RAW_DIR = REPO_ROOT / "data" / "raw" / "default_credit_card_clients"
REPORT_DIR = REPO_ROOT / "reports" / "notebooks" / "kaggle_training_validation"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
SAMPLE_SIZE = 5_000  # keep the notebook quick; set to None to use the full Kaggle dataset
TEST_SIZE = 0.25

print(f"Repository root: {REPO_ROOT}")
print(f"Raw data dir:    {RAW_DIR}")
print(f"Report dir:      {REPORT_DIR}")

## 2. Download the Kaggle dataset

This uses the Kaggle CLI and expects local Kaggle credentials to be configured. The dataset is public and CC0 licensed. If the CSV already exists, the download step is skipped.


In [ ]:
def find_kaggle_cli() -> str:
    candidates = [
        shutil.which("kaggle"),
        str(Path.home() / ".local" / "bin" / "kaggle"),
        "/home/luchopy/.local/bin/kaggle",
    ]
    for candidate in candidates:
        if candidate and Path(candidate).exists():
            return candidate
    raise FileNotFoundError(
        "Kaggle CLI was not found. Install it and configure credentials before running this cell."
    )


def ensure_kaggle_csv() -> Path:
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    csv_candidates = sorted(RAW_DIR.rglob("*.csv"))
    if csv_candidates:
        return csv_candidates[0]

    kaggle = find_kaggle_cli()
    command = [
        kaggle,
        "datasets",
        "download",
        "-d",
        DATASET_SLUG,
        "-p",
        str(RAW_DIR),
        "--unzip",
    ]
    result = subprocess.run(command, capture_output=True, text=True, check=False)
    if result.returncode != 0:
        raise RuntimeError(
            "Kaggle download failed. Confirm credentials with `kaggle datasets list`.\n"
            f"STDOUT:\n{result.stdout}\nSTDERR:\n{result.stderr}"
        )
    csv_candidates = sorted(RAW_DIR.rglob("*.csv"))
    if not csv_candidates:
        raise FileNotFoundError(f"Kaggle download completed but no CSV was found in {RAW_DIR}")
    return csv_candidates[0]


csv_path = ensure_kaggle_csv()
print(f"Using Kaggle CSV: {csv_path}")

## 3. Load and prepare modeling data

The target is `default.payment.next.month`. We train with the numeric columns supplied by the dataset and keep `SEX` as a simple validation segment.


In [ ]:
raw = pd.read_csv(csv_path)
raw.columns = [column.strip() for column in raw.columns]
raw = raw.rename(columns={"default.payment.next.month": "target"})
if "ID" in raw.columns:
    raw = raw.drop(columns=["ID"])
if "target" not in raw.columns:
    raise ValueError(f"Could not find target column. Columns: {list(raw.columns)}")

raw = raw.dropna(subset=["target"]).copy()
raw["target"] = raw["target"].astype(int)
raw["sex_segment"] = (
    "sex_" + raw["SEX"].astype("Int64").astype(str) if "SEX" in raw.columns else "all"
)

if SAMPLE_SIZE is not None and len(raw) > SAMPLE_SIZE:
    raw, _ = train_test_split(
        raw,
        train_size=SAMPLE_SIZE,
        stratify=raw["target"],
        random_state=RANDOM_STATE,
    )
    raw = raw.reset_index(drop=True)

feature_cols = [column for column in raw.columns if column not in {"target", "sex_segment"}]
X = raw[feature_cols]
y = raw["target"]
segments = raw["sex_segment"]

X_train, X_test, y_train, y_test, segment_train, segment_test = train_test_split(
    X,
    y,
    segments,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Rows used:       {len(raw):,}")
print(f"Train rows:      {len(X_train):,}")
print(f"Test rows:       {len(X_test):,}")
print(f"Test bad rate:   {y_test.mean():.2%}")
print(f"Feature columns: {len(feature_cols)}")

## 4. Train Logistic Regression and XGBoost

Both models are intentionally simple. The goal is to demonstrate validation reporting, not to tune a production credit-risk model.


In [ ]:
numeric_features = feature_cols
logistic_model = Pipeline(
    steps=[
        (
            "preprocess",
            ColumnTransformer(
                transformers=[
                    (
                        "numeric",
                        Pipeline(
                            steps=[
                                ("imputer", SimpleImputer(strategy="median")),
                                ("scaler", StandardScaler()),
                            ]
                        ),
                        numeric_features,
                    )
                ],
                remainder="drop",
            ),
        ),
        ("model", LogisticRegression(max_iter=1_000, random_state=RANDOM_STATE)),
    ]
)

negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())
scale_pos_weight = negative_count / max(positive_count, 1)

xgb_model = XGBClassifier(
    n_estimators=50,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.90,
    colsample_bytree=0.90,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    tree_method="hist",
    n_jobs=1,
)

logistic_model.fit(X_train, y_train)
xgb_model.fit(X_train, y_train)

logistic_pd = logistic_model.predict_proba(X_test)[:, 1]
xgb_pd = xgb_model.predict_proba(X_test)[:, 1]

print(f"Logistic test AUC: {roc_auc_score(y_test, logistic_pd):.3f}")
print(f"XGBoost test AUC:  {roc_auc_score(y_test, xgb_pd):.3f}")

## 5. Build CRVK validation samples

CRVK expects already-scored validation data. For each model we create a test-set frame with:

- `target`: observed default flag.
- `pd`: model predicted probability of default.
- `score`: monotonic score derived from PD.
- `sex_segment`: segment used for segment-level validation tables.


In [ ]:
def pd_to_score(pd_values: np.ndarray) -> np.ndarray:
    clipped = np.clip(pd_values, 1e-6, 1 - 1e-6)
    return -np.log(clipped / (1 - clipped))


def make_validation_frame(pd_values: np.ndarray) -> pl.DataFrame:
    return pl.DataFrame(
        {
            "target": y_test.to_numpy(dtype=int),
            "pd": pd_values.astype(float),
            "score": pd_to_score(pd_values).astype(float),
            "sex_segment": segment_test.astype(str).to_numpy(),
        }
    )


validation_frames = {
    "logistic_regression": make_validation_frame(logistic_pd),
    "xgboost": make_validation_frame(xgb_pd),
}

validation_frames["logistic_regression"].head()

## 6. Generate one validation report per model

This is the main training-validation workflow: the report is generated on the held-out test set, not on train data and not as a drift comparison.


In [ ]:
def run_training_validation_report(
    model_key: str, model_name: str, validation_frame: pl.DataFrame
) -> dict[str, object]:
    suite = PDValidationSuite(
        target_col="target",
        pd_col="pd",
        score_col="score",
        segment_cols=["sex_segment"],
        score_direction="lower_is_riskier",
        n_bins=10,
        min_events=20,
        min_non_events=20,
        min_rows=50,
    )
    suite.config.model.name = model_name
    suite.config.model.version = "notebook-demo"
    suite.config.model.horizon = "12m"
    suite.config.report.title = f"{model_name} Training Validation Report"

    result = suite.run(validation_data=validation_frame)
    html_path = REPORT_DIR / f"{model_key}_training_validation_report.html"
    json_path = REPORT_DIR / f"{model_key}_training_validation_metrics.json"
    tables_dir = REPORT_DIR / f"{model_key}_tables"

    result.to_html(html_path)
    result.to_json(json_path)
    result.to_tables(tables_dir)

    return {
        "model": model_name,
        "status": result.status.value,
        "auc": result.metrics["auc"].value,
        "gini": result.metrics["gini"].value,
        "ks": result.metrics["ks"].value,
        "brier": result.metrics["brier"].value,
        "log_loss": result.metrics["log_loss"].value,
        "ece": result.metrics["ece"].value,
        "oe_ratio": result.metrics["oe_ratio"].value,
        "html_report": str(html_path),
        "json_metrics": str(json_path),
    }


summaries = [
    run_training_validation_report(
        "logistic_regression",
        "Logistic Regression",
        validation_frames["logistic_regression"],
    ),
    run_training_validation_report("xgboost", "XGBoost", validation_frames["xgboost"]),
]

summary = pd.DataFrame(summaries)
summary

## 7. Report locations

Open the HTML files below in a browser to review the full CRVK validation evidence for each trained model.


In [ ]:
for item in summaries:
    print(f"{item['model']}: {Path(item['html_report']).resolve()}")